### Data Profiling 

Early data profiling within the raw dataset

In [0]:
# Read from the bronze Delta table written by 00-Input-Data
df = spark.table("students_data.`team4-chicago`.food_inspections_bronze")

print(f"Loaded from bronze table: {df.count():,} rows, {len(df.columns)} columns")

Null counts + distinct counts + type validation + category check

In [0]:
from pyspark.sql import functions as F

total_rows = df.count()

# Null counts and percentages per column
null_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]
null_counts = df.select(null_exprs).collect()[0]

print("=" * 65)
print(f"{'COLUMN':<25} {'NULLS':>10} {'% NULL':>10}  {'DTYPE':<15}")
print("=" * 65)
for field in df.schema.fields:
    c = field.name
    n = null_counts[c]
    pct = (n / total_rows) * 100
    flag = " ⚠️" if pct > 5 else ""
    print(f"{c:<25} {n:>10,} {pct:>9.2f}%  {str(field.dataType):<15}{flag}")
print(f"\nTotal rows: {total_rows:,}")

# Distinct value counts
print("\n" + "=" * 45)
print(f"{'COLUMN':<25} {'DISTINCT':>10}")
print("=" * 45)
distinct_exprs = [F.countDistinct(F.col(c)).alias(c) for c in df.columns]
distinct_counts = df.select(distinct_exprs).collect()[0]
for c in df.columns:
    print(f"{c:<25} {distinct_counts[c]:>10,}")

# Type validation: check string columns that should be numeric
print("\n" + "=" * 65)
print("TYPE VALIDATION: String columns checked for numeric content")
print("=" * 65)
string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]
for c in string_cols:
    non_null = df.filter(F.col(c).isNotNull())
    non_numeric = non_null.filter(~F.col(c).rlike(r"^-?\d+\.?\d*$")).count()
    total_non_null = non_null.count()
    if total_non_null > 0:
        numeric_pct = ((total_non_null - non_numeric) / total_non_null) * 100
        flag = " → candidate for CAST to numeric" if numeric_pct > 90 else ""
        print(f"{c:<25} {numeric_pct:>6.1f}% numeric values{flag}")

# Categorical columns - top values preview
print("\n" + "=" * 65)
print("CATEGORICAL COLUMNS: Top 5 values")
print("=" * 65)
categorical_candidates = [c for c in string_cols if distinct_counts[c] < 50]
for c in categorical_candidates:
    top_vals = (
        df.filter(F.col(c).isNotNull())
        .groupBy(c)
        .count()
        .orderBy(F.desc("count"))
        .limit(5)
        .collect()
    )
    vals_str = ", ".join([f"{row[c]} ({row['count']:,})" for row in top_vals])
    print(f"\n  {c}: {vals_str}")

Validity check of Fail result & Null Violations

In [0]:
from pyspark.sql import functions as F

# Null violations where result is Pass = expected, no concern
null_viol_pass = df.filter(F.col("Violations").isNull() & (F.col("Results") == "Pass")).count()

# Null violations where result is Fail = suspicious, flag these
null_viol_fail = df.filter(F.col("Violations").isNull() & (F.col("Results") == "Fail"))
null_viol_fail_count = null_viol_fail.count()

print(f"Null Violations + Pass result (expected):  {null_viol_pass:,}")
print(f"Null Violations + Fail result (suspicious): {null_viol_fail_count:,}")

# Show all IDs that failed but have no violation details
print(f"\n--- Inspection IDs with Fail result but NULL Violations ---")
display(
    null_viol_fail
    .select("Inspection_ID", "Violations", "DBA_Name", "Inspection_Date", "Results")
    .orderBy("Inspection_Date")
)

City + state + zip code validity for Chicago

In [0]:
from pyspark.sql import functions as F

# Define what valid Chicago location looks like
chicago_zip_min = 60601
chicago_zip_max = 60827

df_location = df.withColumn(
    "city_is_chicago", F.upper(F.trim(F.col("City"))) == "CHICAGO"
).withColumn(
    "state_is_il", F.upper(F.trim(F.col("State"))) == "IL"
).withColumn(
    "zip_is_chicago", (F.col("Zip") >= chicago_zip_min) & (F.col("Zip") <= chicago_zip_max)
)

# Flag: mismatch if any of the three fields contradict each other
# All agree Chicago → fine | All agree non-Chicago → fine | Mixed → flag
df_flagged = df_location.withColumn(
    "location_flag",
    F.when(
        # All three say Chicago → consistent, no flag
        (F.col("city_is_chicago") & F.col("state_is_il") & F.col("zip_is_chicago")),
        F.lit("OK")
    ).when(
        # Any field is null → can't validate
        F.col("City").isNull() | F.col("State").isNull() | F.col("Zip").isNull(),
        F.lit("MISSING")
    ).otherwise(
        # At least one field disagrees with the others → mismatch
        F.lit("MISMATCH")
    )
)

# Summary counts
print("=" * 50)
print("LOCATION CONSISTENCY SUMMARY")
print("=" * 50)
display(
    df_flagged.groupBy("location_flag").count()
    .orderBy(F.desc("count"))
)

# Show the mismatched rows so we can see what's wrong
print("\n--- MISMATCH details ---")
display(
    df_flagged
    .filter(F.col("location_flag") == "MISMATCH")
    .select("Inspection_ID", "DBA_Name", "City", "State", "Zip",
            "city_is_chicago", "state_is_il", "zip_is_chicago")
    .orderBy("Inspection_ID")
)

Check all non-chicago cities

In [0]:
from pyspark.sql import functions as F

non_chicago = df.filter(F.upper(F.trim(F.col("City"))) != "CHICAGO")

print(f"Rows where City != CHICAGO: {non_chicago.count():,}")
display(
    non_chicago
    .select("Inspection_ID", "DBA_Name", "City", "State", "Zip", "Address")
    .orderBy("City")
)

Count of columns within the non-chicago range

In [0]:
from pyspark.sql import functions as F

# If State = IL and Zip is in the 60xxx range, correct City to CHICAGO
df = df.withColumn(
    "City",
    F.when(
        (F.upper(F.trim(F.col("State"))) == "IL") &
        (F.col("Zip").between(60000, 60999)) &
        (F.upper(F.trim(F.col("City"))) != "CHICAGO"),
        F.lit("CHICAGO")
    ).otherwise(F.col("City"))
)

print("City column updated in df. Rows where City != CHICAGO now:")
print(df.filter(F.upper(F.trim(F.col("City"))) != "CHICAGO").count())

Display non-chicago cities


In [0]:
from pyspark.sql import functions as F

non_chicago = df.filter(F.upper(F.trim(F.col("City"))) != "CHICAGO")

print(f"Rows where City != CHICAGO: {non_chicago.count():,}")
display(
    non_chicago
    .select("Inspection_ID", "DBA_Name", "City", "State", "Zip", "Address")
    .orderBy("City")
)

Drop verified non-chicago cities

In [0]:
from pyspark.sql import functions as F

rows_before = df.count()

# Drop rows where City != CHICAGO
df = df.filter(F.upper(F.trim(F.col("City"))) == "CHICAGO")

rows_after = df.count()
chicago_count = df.filter(F.upper(F.trim(F.col("City"))) == "CHICAGO").count()

print(f"Rows before drop:        {rows_before:,}")
print(f"Rows after drop:         {rows_after:,}")
print(f"Rows dropped:            {rows_before - rows_after:,}")
print(f"Rows where City=CHICAGO: {chicago_count:,}")
print(f"Match: {rows_after == chicago_count}")

understand outliers for risk column

In [0]:
from pyspark.sql import functions as F

risk_all = df.filter(F.col("Risk") == "All")

print(f"Rows where Risk = 'All': {risk_all.count():,}")
display(risk_all)

Risk null = unknown

In [0]:
from pyspark.sql import functions as F

valid_risks = ["Risk 1 (High)", "Risk 2 (Medium)", "Risk 3 (Low)"]

# Replace Risk with "Unknown" if null, "All", or not one of the 3 standard levels
df = df.withColumn(
    "Risk",
    F.when(F.col("Risk").isin(valid_risks), F.col("Risk"))
    .otherwise(F.lit("Unknown"))
)

# Replace Facility Type with "Unknown" if null
df = df.withColumn(
    "Facility_Type",
    F.when(F.col("Facility_Type").isNull(), F.lit("Unknown"))
    .otherwise(F.col("Facility_Type"))
)

# Verify
print("Risk distribution after fix:")
display(df.groupBy("Risk").count().orderBy(F.desc("count")))

print(f"\nFacility Type nulls remaining: {df.filter(F.col('Facility_Type').isNull()).count()}")

## Summary of Findings

### Dataset Overview
* **Source:** `Food_Inspections.csv` (322.7 MB) from `/Volumes/students_data/team4-chicago/chicago-restaurants`
* **Original row count:** 307,282 rows | **17 columns**
* **Final row count after profiling:** 307,084 rows

---

### Key Findings

| Area | Finding | Severity |
| --- | --- | --- |
| **Violations** | 28% null (86,154 rows). Of these, 39,109 are Pass results (expected). **3,609 are Fail results with no violation details** — data entry gaps. | High |
| **Latitude / Longitude** | Stored as strings but \~96% numeric content. Candidates for casting to `double` in the silver layer. | Medium |
| **Location** | Composite string column `(lat, lon)` — redundant if Lat/Long are properly cast. | Low |
| **City** | 375 rows were not "CHICAGO". Mix of typos (`CCHICAGO`, `CHCHICAGO`, `312CHICAGO`, `CHICAGO.`) and genuinely non-Chicago suburbs. | Medium |
| **Risk** | 84 rows had `Risk = "All"` — almost all were pending license inspections with `Results = "Not Ready"`. | Low |
| **Facility Type** | 5,322 nulls (1.7%) and 520 distinct values — some may be duplicates due to inconsistent naming. | Medium |
| **Zip** | Range from 10014 to 91706 — some clearly outside Chicago's 606xx–608xx range. | Medium |
| **State** | 6 distinct values, but 307,195 of 307,282 were IL. Non-IL entries were edge cases (IN, CA, WI, CO, NY). | Low |

---

### What Was Amended

1. **City column corrected** — Where `State = IL` and `Zip` was in the `60xxx` range, City was overwritten to `"CHICAGO"`. This fixed 355 typos and mismatched suburb entries.
2. **Non-Chicago rows dropped** — 198 rows where City, State, and Zip were all consistently non-Chicago (e.g. Baldwin Park CA, Merriville IN) were removed.
3. **Risk standardised** — Values of `"All"`, `null`, or anything outside the 3 standard levels (`Risk 1 (High)`, `Risk 2 (Medium)`, `Risk 3 (Low)`) were replaced with `"Unknown"` (169 rows).
4. **Facility Type nulls filled** — All null Facility Type values replaced with `"Unknown"`.

---

### What to Keep an Eye on Going Forward (Silver Layer)

* **Violations column (28% null):** Decide how to handle the 3,609 Fail records with no violation text — drop, flag, or fill with a placeholder like `"UNKNOWN VIOLATION"`.
* **Latitude / Longitude:** Cast from string to `double` and validate values fall within Chicago's bounding box (lat 41.6–42.1, lon \-87.9 to \-87.5). Currently 1,020–1,030 nulls in these columns.
* **Location column:** Consider dropping entirely — it is redundant with Latitude and Longitude.
* **Facility Type (520 distinct values):** Likely contains duplicates from inconsistent casing or naming (e.g. "Restaurant" vs "RESTAURANT"). Needs grouping/standardisation.
* **License # column:** Min value is 0 — likely invalid. Investigate how many zero or suspiciously low values exist.
* **Inspection Date:** Check for future dates or unreasonably old dates that could indicate data load issues.
* **DBA Name / AKA Name:** Check for leading/trailing whitespace and inconsistent casing that could cause join issues downstream.
* **Duplicate detection:** Verify `Inspection ID` uniqueness. Also check for near-duplicates (same `License #` + `Inspection Date` + `Inspection Type`).